# Nifty 50 Portfolio VaR & Expected Shortfall Analysis

This notebook builds a multi-stock Value at Risk (VaR) model for a portfolio of Nifty 50
constituents, using three standard methods — Historical Simulation, Parametric
(Variance-Covariance), and Monte Carlo Simulation — and computes Expected Shortfall (ES)
alongside each, since VaR alone doesn't say anything about the severity of losses beyond
the threshold.

**Why a portfolio rather than just the index:** modeling a basket of individual stocks
forces the use of a full covariance matrix and correlation structure, which is the same
underlying mechanic used in institutional portfolio risk tools (e.g. BarraOne, RiskMetrics)
rather than a single-series exercise.

**Scope and assumptions (stated explicitly, since every risk model lives or dies on its assumptions):**
- 10 Nifty 50 constituents spanning banking, IT, energy, FMCG, and telecom
- Equal weighting (10% each) — a simplifying assumption; cap-weighting or
  factor-based weighting is a natural next step
- Returns are simple daily returns, not log returns
- Parametric VaR assumes normally distributed returns, which is known to
  understate tail risk for equities (fat tails) — this is discussed at the end
- Lookback window: ~3 years of daily data


## 1. Setup and Data

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(42)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


In [ ]:
# Portfolio: 10 Nifty 50 constituents across sectors, equal-weighted.
# Swap in cap-weights or any other allocation by editing the `weights` dict.

tickers = {
    'RELIANCE.NS':  'Energy',
    'TCS.NS':       'IT',
    'INFY.NS':      'IT',
    'HDFCBANK.NS':  'Banking',
    'ICICIBANK.NS': 'Banking',
    'SBIN.NS':      'Banking',
    'ITC.NS':       'FMCG',
    'HINDUNILVR.NS':'FMCG',
    'LT.NS':        'Industrials',
    'BHARTIARTL.NS':'Telecom',
}

n_assets = len(tickers)
weights = pd.Series(1 / n_assets, index=tickers.keys())  # equal-weighted
portfolio_value = 10_00_00_000  # INR 10 crore notional, change as needed

print(f"Portfolio of {n_assets} stocks, equal-weighted at {weights.iloc[0]:.1%} each")
print(f"Notional portfolio value: Rs. {portfolio_value:,.0f}")


In [ ]:
# Pull ~3 years of daily adjusted close prices
prices = yf.download(list(tickers.keys()), start='2022-01-01', end='2024-12-31', auto_adjust=True)['Close']
prices = prices.dropna()

returns = prices.pct_change().dropna()
returns.tail()


## 2. Portfolio Return Series and Covariance Structure

Two things matter here for the methods below: the weighted daily portfolio return series
(used directly by Historical Simulation) and the asset covariance matrix (used by both
Parametric and Monte Carlo methods).

In [ ]:
w = weights[returns.columns].values  # align order
mean_returns = returns.mean().values
cov_matrix = returns.cov().values

# Weighted daily portfolio return series
portfolio_returns = returns.values @ w

port_mean = portfolio_returns.mean()
port_std = portfolio_returns.std()

print(f"Annualized portfolio return: {port_mean*252:.2%}")
print(f"Annualized portfolio volatility: {port_std*np.sqrt(252):.2%}")

plt.figure(figsize=(10, 4))
plt.plot(returns.index, (1 + pd.Series(portfolio_returns, index=returns.index)).cumprod())
plt.title('Cumulative Portfolio Value (Rs. 1 invested)')
plt.ylabel('Growth multiple')
plt.tight_layout()
plt.show()


## 3. Method 1 — Historical Simulation VaR & Expected Shortfall

No distributional assumption: we directly take the empirical quantile of actual past
portfolio returns. This captures real fat tails and skew in the data but is fully
backward-looking — it implicitly assumes the future will resemble the historical window.

In [ ]:
def historical_var_es(port_returns, confidence, value):
    """Historical Simulation VaR and Expected Shortfall.

    VaR is the negative of the (1-confidence) percentile of the return distribution.
    ES is the average loss in the tail beyond that VaR threshold.
    """
    var_pct = -np.percentile(port_returns, (1 - confidence) * 100)
    tail_losses = port_returns[port_returns <= -var_pct]
    es_pct = -tail_losses.mean()
    return var_pct * value, es_pct * value

hist_results = {}
for conf in [0.95, 0.99]:
    var, es = historical_var_es(portfolio_returns, conf, portfolio_value)
    hist_results[conf] = (var, es)
    print(f"Historical  {conf:.0%} VaR: Rs. {var:,.0f}   |   ES: Rs. {es:,.0f}")


## 4. Method 2 — Parametric (Variance-Covariance) VaR & Expected Shortfall

Assumes portfolio returns are normally distributed. Portfolio variance is derived
analytically from the asset covariance matrix (`w^T Σ w`), so this method is fast and
closed-form — but it will understate risk whenever real returns have fatter tails than
the normal distribution, which is the norm for equities, not the exception.

In [ ]:
def parametric_var_es(mean, std, confidence, value):
    """Parametric (variance-covariance) VaR and ES under a normality assumption."""
    z = norm.ppf(1 - confidence)
    var_pct = -(mean + z * std)
    # Closed-form ES for a normal distribution
    es_pct = -(mean - std * norm.pdf(z) / (1 - confidence))
    return var_pct * value, es_pct * value

# Portfolio variance derived from the full covariance matrix: w^T * Sigma * w
portfolio_variance = w @ cov_matrix @ w
portfolio_std_analytical = np.sqrt(portfolio_variance)

param_results = {}
for conf in [0.95, 0.99]:
    var, es = parametric_var_es(port_mean, portfolio_std_analytical, conf, portfolio_value)
    param_results[conf] = (var, es)
    print(f"Parametric  {conf:.0%} VaR: Rs. {var:,.0f}   |   ES: Rs. {es:,.0f}")


## 5. Method 3 — Monte Carlo Simulation VaR & Expected Shortfall

Simulates thousands of correlated return scenarios using a Cholesky decomposition of the
covariance matrix (so the simulated assets retain the same correlation structure as the
historical data), then takes the empirical quantile of the simulated portfolio P&L. This
still assumes the same covariance matrix is forward-looking, but it doesn't force a
normal-distribution shortcut at the portfolio level the way the parametric method does,
and it generalizes more easily to non-linear payoffs (options, etc.) later.

In [ ]:
def monte_carlo_var_es(mean_returns, cov_matrix, weights, confidence, value, n_sims=50_000, seed=42):
    """Monte Carlo VaR and ES via correlated return simulation (Cholesky decomposition)."""
    rng = np.random.default_rng(seed)
    L = np.linalg.cholesky(cov_matrix)
    z = rng.standard_normal((n_sims, len(weights)))
    simulated_returns = mean_returns + z @ L.T
    simulated_port_returns = simulated_returns @ weights

    var_pct = -np.percentile(simulated_port_returns, (1 - confidence) * 100)
    tail_losses = simulated_port_returns[simulated_port_returns <= -var_pct]
    es_pct = -tail_losses.mean()
    return var_pct * value, es_pct * value, simulated_port_returns

mc_results = {}
mc_sims = {}
for conf in [0.95, 0.99]:
    var, es, sims = monte_carlo_var_es(mean_returns, cov_matrix, w, conf, portfolio_value)
    mc_results[conf] = (var, es)
    mc_sims[conf] = sims
    print(f"Monte Carlo {conf:.0%} VaR: Rs. {var:,.0f}   |   ES: Rs. {es:,.0f}")


## 6. Side-by-Side Comparison

In [ ]:
comparison = pd.DataFrame({
    'Historical VaR':  [hist_results[c][0] for c in [0.95, 0.99]],
    'Historical ES':   [hist_results[c][1] for c in [0.95, 0.99]],
    'Parametric VaR':  [param_results[c][0] for c in [0.95, 0.99]],
    'Parametric ES':   [param_results[c][1] for c in [0.95, 0.99]],
    'Monte Carlo VaR': [mc_results[c][0] for c in [0.95, 0.99]],
    'Monte Carlo ES':  [mc_results[c][1] for c in [0.95, 0.99]],
}, index=['95%', '99%']).round(0)

comparison


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(portfolio_returns * portfolio_value, bins=60, alpha=0.6, label='Historical P&L distribution', color='steelblue')
ax.axvline(-hist_results[0.95][0], color='orange', linestyle='--', label='95% Historical VaR')
ax.axvline(-hist_results[0.95][1], color='red', linestyle='--', label='95% Historical ES')
ax.axvline(-param_results[0.95][0], color='green', linestyle=':', label='95% Parametric VaR')
ax.set_title('Portfolio Daily P&L Distribution with VaR / ES Markers (95%)')
ax.set_xlabel('Daily P&L (Rs.)')
ax.legend()
plt.tight_layout()
plt.show()


## 7. Interpretation and Limitations

A few things worth calling out explicitly rather than letting the numbers speak for
themselves:

- **Where the methods agree vs diverge**: if Parametric VaR comes in noticeably lower
  than Historical VaR at the 99% level, that's a signal the actual return distribution
  has fatter tails than the normal assumption accounts for — a well-known property of
  equity returns, and exactly the kind of thing institutional risk teams flag in model
  validation.
- **Equal weighting is a simplification.** Cap-weighting the portfolio to match actual
  Nifty 50 index weights would change both the volatility and the diversification
  benefit captured by the covariance matrix.
- **All three methods use the same historical covariance matrix** as an input (directly,
  for Parametric and Monte Carlo; indirectly, for Historical). None of them account for
  the possibility that correlations themselves spike during stress periods, which is a
  known failure mode of VaR during crises.
- **VaR doesn't describe the magnitude of tail losses** — that's precisely what
  Expected Shortfall is for, which is why Basel III and most modern risk frameworks
  have shifted toward ES as the primary risk metric rather than VaR alone.

## 8. Natural Next Steps

Logical extensions from here, roughly in order of effort:
1. **Backtesting** (Kupiec proportion-of-failures test, Christoffersen independence
   test) to check whether the model's VaR breaches actually occur at the expected
   frequency historically.
2. **GARCH-based volatility** instead of the static covariance matrix, so the model
   reacts to volatility clustering rather than assuming constant variance.
3. **Cap-weighting** the portfolio to match actual Nifty 50 index composition.
4. **Factor decomposition** (e.g., market beta, sector factors) to attribute portfolio
   risk to systematic sources rather than treating each stock independently.
